# Abnormality distribution by kernel and scanner

Run this **right after `make_worklist.py`** -- it needs no images, only what that script already downloaded
(the CT-RATE metadata, the label CSVs, and the run's `worklist.csv`) plus the sharp / soft table
`configs/kernel_classes.csv`.

It answers three questions *before* the long download:

1. **What does the run actually take?** How many volumes per pool, kernel class and scanner, and how many train scans are
   skipped because they have no reconstruction of the wanted kernel.
2. **Do the labels depend on the scanner or the kernel?** If a finding is much more common on one scanner, a model can
   learn "this looks like scanner X" instead of the finding (shortcut learning) -- and a model trained on one kernel is
   partly a model of one group of scanners.
3. **Are the skipped scans different from the kept ones?** Skipping them is only harmless if they are not a distinct group.

The labels are CT-RATE's *predicted* labels (parsed from the radiology reports by a model), not human-checked ground truth.
Labels belong to a scan, so both reconstructions of a scan carry the same labels; what differs between kernels is *which
scans* have that kernel.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# run from the repository root, wherever the notebook was opened
root = Path.cwd().resolve()
while not (root / "configs" / "preprocessing.yaml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)

from ct_preprocessing.config import load_config
from ct_preprocessing.ingest.ctrate import load_metadata
from ct_preprocessing.ingest.labels import label_column, load_ctrate_labels
from ct_preprocessing.ingest.worklist import read_worklist
from ct_preprocessing.kernels import add_kernel_columns, load_kernel_table
from ct_preprocessing.manifest import parse_volume_id, strip_nifti_ext
from ct_preprocessing.runs import list_runs, resolve_run

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 1. Pick the run

`RUN = None` uses the only run on disk; with several, name one. `CONFIG` is the settings file the scripts used.

In [ ]:
CONFIG = "configs/preprocessing.yaml"
SOURCE = "ctrate"
RUN = None  # e.g. "train-sharp"

cfg = load_config(CONFIG)
print("runs of", SOURCE + ":", list_runs(cfg.paths, SOURCE) or "none yet -- run make_worklist.py first")
run = resolve_run(cfg.paths, SOURCE, RUN)
worklist = read_worklist(run.worklist_path)
table = load_kernel_table(cfg.paths.kernel_table)
train_kernel = (pd.read_json(run.info_path, typ="series").get("settings") or {}).get("train_kernel", "sharp")
print(f"run {run.name!r}: {len(worklist)} volumes in {worklist.chunk.nunique()} chunks; train kernel = {train_kernel}")

In [ ]:
meta = load_metadata(cfg.paths)
labels = load_ctrate_labels(cfg.paths)
if labels is None:
    raise SystemExit("no label files in " + cfg.paths.metadata_dir + " -- make_worklist.py downloads them (or copy "
                     "train_predicted_labels.csv / valid_predicted_labels.csv there by hand)")
label_cols = [c for c in labels.columns if c != "volume_id"]
label_names = {c: label_column(c) for c in label_cols}

# every train / valid volume of the metadata, with its scanner, kernel and kernel class
frames = []
for pool, df in meta.items():
    ids = df["VolumeName"].map(strip_nifti_ext)
    parsed = pd.DataFrame([parse_volume_id(v) for v in ids])
    parsed["manufacturer"], parsed["_kernel"] = df["Manufacturer"].values, df["ConvolutionKernel"].values
    frames.append(parsed)
catalog = add_kernel_columns(pd.concat(frames, ignore_index=True), table, manufacturer_col="manufacturer", kernel_col="_kernel")
catalog = catalog.drop(columns="_kernel").merge(labels, on="volume_id", how="left")
print(f"{len(catalog)} volumes in the metadata; {catalog[label_cols[0]].notna().sum()} have labels")

## 2. What the run takes

The worklist always takes the whole valid pool (the test set, every reconstruction) and one reconstruction of the chosen kernel
per train scan.

In [ ]:
wl = worklist.merge(labels, on="volume_id", how="left")
print("volumes by pool and kernel class")
display(pd.crosstab(wl.source_split, wl.kernel_class, margins=True))
print("volumes by scanner and kernel class")
display(pd.crosstab(wl.manufacturer, wl.kernel_class, margins=True))

In [ ]:
train = catalog[catalog.source_split == "train"]
scans = train.groupby(["patient_id", "scan_id"]).agg(
    has_wanted=("kernel_class", lambda s: (s == train_kernel).any()),
    n_recon=("volume_id", "size"),
    manufacturer=("manufacturer", "first"),
).reset_index()
skipped = scans[~scans.has_wanted]
patients_with_none = int((~scans.groupby("patient_id").has_wanted.any()).sum())
print(f"{len(scans)} train scans; {int(scans.has_wanted.sum())} have a {train_kernel} reconstruction, "
      f"{len(skipped)} are skipped ({100 * len(skipped) / len(scans):.1f}%) -- {patients_with_none} patients have no {train_kernel} scan at all")
print("\nskipped scans by scanner (share of that scanner's scans):")
share = (skipped.manufacturer.value_counts() / scans.manufacturer.value_counts()).dropna().mul(100).round(1)
display(pd.DataFrame({"skipped": skipped.manufacturer.value_counts(), "of_scans": scans.manufacturer.value_counts(), "percent": share}).dropna())

## 3. Do the labels depend on the scanner?

Prevalence (percent of scans with the finding) in the train volumes the run takes, per scanner. A large spread means the
scanner and the label go together.

In [ ]:
taken = wl[wl.source_split == "train"].copy()
by_scanner = taken.groupby("manufacturer")[label_cols].mean().mul(100).round(1)
by_scanner["scans"] = taken.groupby("manufacturer").size()
by_scanner = by_scanner[by_scanner.scans >= 30]  # a few scans say nothing
overall = taken[label_cols].mean().mul(100).round(1)

spread = (by_scanner[label_cols].max() - by_scanner[label_cols].min()).round(1)
summary = pd.DataFrame({"overall_%": overall, "lowest_scanner_%": by_scanner[label_cols].min(),
                        "highest_scanner_%": by_scanner[label_cols].max(), "spread": spread}).sort_values("spread", ascending=False)
display(summary)
print("scanners compared:", ", ".join(f"{m} ({n})" for m, n in by_scanner.scans.items()))

In [ ]:
from scipy.stats import chi2_contingency

rows = []
for col in label_cols:
    counts = taken.dropna(subset=[col]).groupby("manufacturer")[col].agg(["sum", "count"])
    counts = counts[counts["count"] >= 30]
    table2 = np.array([counts["sum"], counts["count"] - counts["sum"]])
    p = chi2_contingency(table2)[1] if table2.shape[1] > 1 and table2.min() >= 0 else np.nan
    rows.append({"label": col, "p_value": p, "spread_points": float(spread[col])})
tests = pd.DataFrame(rows).sort_values("p_value")
tests["flag"] = np.where((tests.p_value < 0.001) & (tests.spread_points >= 5), "depends on scanner", "")
display(tests.reset_index(drop=True))
print(f"{(tests.flag != '').sum()} of {len(tests)} labels differ clearly between scanners "
      "(p < 0.001 and at least 5 percentage points) -- with this many scans a small spread can be significant, so read the spread too")

In [ ]:
import matplotlib.pyplot as plt

top = list(summary.index[:6])
ax = by_scanner[top].T.plot.bar(figsize=(10, 4), width=0.8)
ax.set_ylabel("% of scans with the finding")
ax.set_title("The six findings that differ most between scanners (train volumes in the run)")
ax.legend(title="scanner", fontsize=8)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 4. Are the skipped scans different from the kept ones?

If scans without the wanted kernel have a different mix of findings, skipping them changes what the model sees.

In [ ]:
lab_train = train.drop_duplicates(["patient_id", "scan_id"]).merge(scans[["patient_id", "scan_id", "has_wanted"]], on=["patient_id", "scan_id"])
cmp = lab_train.groupby("has_wanted")[label_cols].mean().T.mul(100).round(1)
cmp.columns = ["skipped_%", "kept_%"] if list(cmp.columns) == [False, True] else cmp.columns
cmp["difference"] = (cmp.iloc[:, 0] - cmp.iloc[:, 1]).round(1)
display(cmp.sort_values("difference"))
print(f"{len(skipped)} skipped scans against {int(scans.has_wanted.sum())} kept: a difference of several points on a finding means "
      "the skipped scans are a distinct group, but they are a small share of the data")

## 5. The test pool, by kernel

The test pool takes every reconstruction, so a model can be tested on sharp and on soft. Labels are per scan, so the two kernels
carry the same labels; what to look at is how many volumes and scanners each view has.

In [ ]:
test = catalog[catalog.source_split == "valid"]
print("test volumes:", len(test), "| scans:", len(test.drop_duplicates(["patient_id", "scan_id"])), "| patients:", test.patient_id.nunique())
display(pd.crosstab(test.manufacturer, test.kernel_class, margins=True))
print("prevalence (% of volumes) by kernel class -- labels are per scan, so a gap here only means the two kernels sit on different scans")
display(test.groupby("kernel_class")[label_cols].mean().T.mul(100).round(1))

In [ ]:
manifest_file = run.manifest_path
if manifest_file.exists():
    manifest = pd.read_csv(manifest_file)
    print("manifest of this run:", len(manifest), "volumes")
    display(pd.crosstab(manifest.split, manifest.kernel_class, margins=True))
else:
    print("no manifest yet -- it appears after ingest.py and merge_manifests.py")

## How to read this

- **Few labels flagged, small spreads:** scanner and finding are mostly independent -- sharp-only training is a clean choice.
- **A finding that follows a scanner** (for example one scanner with far more pleural effusion): report accuracy per scanner group
  as well as overall, and consider balancing the sampling by scanner.
- **Skipped scans clearly different:** consider whether to keep them (`train_kernel` takes only scans that have the kernel; a separate run with the other kernel can add them).